# Phase 4: Data Pipeline — Dataset & DataLoader

## What you'll learn
- Custom `Dataset` class
- `DataLoader` for batching, shuffling, parallel loading
- `torchvision.transforms` for augmentation
- Built-in datasets (MNIST, CIFAR10)
- Train/val splits
- Handling imbalanced data

---

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader, random_split, WeightedRandomSampler
import torchvision
import torchvision.transforms as transforms
import numpy as np

## 4.1 Custom Dataset

Every dataset in PyTorch inherits from `Dataset` and must implement:
1. `__len__()` — return dataset size
2. `__getitem__(idx)` — return one sample (features, label)

In [ ]:
class SyntheticDataset(Dataset):
    """A simple synthetic dataset for demonstration."""

    def __init__(self, n_samples=1000, n_features=10):
        self.X = torch.randn(n_samples, n_features)
        # Binary labels based on sum of features
        self.y = (self.X.sum(dim=1) > 0).long()

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

dataset = SyntheticDataset(1000, 10)
print(f"Dataset size: {len(dataset)}")

# Access a single sample
features, label = dataset[0]
print(f"Features shape: {features.shape}")
print(f"Label: {label}")

## 4.2 DataLoader — Batching & Shuffling

`DataLoader` wraps a `Dataset` and provides:
- **Batching** — groups samples into mini-batches
- **Shuffling** — randomizes order each epoch
- **Parallel loading** — `num_workers` for multi-process data loading
- **Pinned memory** — faster CPU→GPU transfer

In [ ]:
loader = DataLoader(
    dataset,
    batch_size=32,
    shuffle=True,       # shuffle every epoch
    num_workers=0,      # 0 = main process (use 2-4 on real projects)
    drop_last=False     # keep last incomplete batch
)

# Iterate through one epoch
for batch_idx, (features, labels) in enumerate(loader):
    if batch_idx == 0:
        print(f"Batch features: {features.shape}")  # (32, 10)
        print(f"Batch labels: {labels.shape}")       # (32,)

print(f"Total batches: {len(loader)}")

## 4.3 Transforms — Data Augmentation

Transforms preprocess and augment data. `torchvision.transforms` provides common image transforms.

Use `transforms.Compose` to chain multiple transforms.

In [ ]:
# Common transform pipeline for training images
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),                    # PIL/numpy → tensor, scales to [0,1]
    transforms.Normalize((0.5,), (0.5,))      # normalize to [-1, 1]
])

# Simpler pipeline for validation (no augmentation)
val_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

print("Train transform:", train_transform)
print("\nVal transform:", val_transform)

## 4.4 Built-in Datasets

`torchvision.datasets` provides popular datasets that auto-download.

In [ ]:
# MNIST — handwritten digits (28x28 grayscale)
mnist_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))  # MNIST mean/std
])

train_dataset = torchvision.datasets.MNIST(
    root='./data',
    train=True,
    download=True,
    transform=mnist_transform
)

test_dataset = torchvision.datasets.MNIST(
    root='./data',
    train=False,
    download=True,
    transform=mnist_transform
)

print(f"Train: {len(train_dataset)} samples")
print(f"Test:  {len(test_dataset)} samples")

img, label = train_dataset[0]
print(f"Image shape: {img.shape}")  # (1, 28, 28)
print(f"Label: {label}")

## 4.5 Train/Validation Split

Use `random_split` to split a dataset into train and validation sets.

In [ ]:
full_dataset = SyntheticDataset(1000, 10)

# 80/20 split
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size

train_set, val_set = random_split(
    full_dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42)  # reproducible
)

print(f"Train: {len(train_set)}, Val: {len(val_set)}")

# Create loaders
train_loader = DataLoader(train_set, batch_size=32, shuffle=True)
val_loader = DataLoader(val_set, batch_size=32, shuffle=False)

## 4.6 Handling Imbalanced Data

When classes are imbalanced, use `WeightedRandomSampler` to oversample minority classes.

In [ ]:
# Simulate imbalanced dataset: 900 class-0, 100 class-1
labels = torch.cat([torch.zeros(900), torch.ones(100)]).long()
features = torch.randn(1000, 10)

# Compute sample weights (inverse of class frequency)
class_counts = torch.bincount(labels)
class_weights = 1.0 / class_counts.float()
sample_weights = class_weights[labels]

print(f"Class counts: {class_counts}")
print(f"Class weights: {class_weights}")

# Create sampler
sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

# Use sampler in DataLoader (can't use shuffle with sampler)
class SimpleDataset(Dataset):
    def __init__(self, features, labels):
        self.features = features
        self.labels = labels
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]

balanced_loader = DataLoader(
    SimpleDataset(features, labels),
    batch_size=32,
    sampler=sampler
)

# Verify balance
all_labels = []
for _, batch_labels in balanced_loader:
    all_labels.append(batch_labels)
all_labels = torch.cat(all_labels)
print(f"\nSampled class distribution: {torch.bincount(all_labels)}")

## 4.7 Custom Dataset with File Loading

Real-world pattern: load data from disk on-the-fly (memory efficient).

In [ ]:
# Example: CSV-based dataset (pattern — won't run without actual file)
class CSVDataset(Dataset):
    def __init__(self, csv_path, transform=None):
        import pandas as pd
        self.data = pd.read_csv(csv_path)
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        features = torch.tensor(row[:-1].values, dtype=torch.float32)
        label = torch.tensor(row.iloc[-1], dtype=torch.long)
        if self.transform:
            features = self.transform(features)
        return features, label

print("CSVDataset class defined (template for real projects)")

## ✅ Phase 4 Summary

You now know how to:
- Create custom `Dataset` classes
- Use `DataLoader` for efficient batching and shuffling
- Apply transforms for data augmentation
- Load built-in datasets (MNIST, CIFAR10)
- Split data into train/val sets
- Handle class imbalance with `WeightedRandomSampler`

**Next → Phase 5: Training Loop Mastery**